# CURE-Rec — remaining reviewer actions

This notebook is the clean execution surface for the remaining reviewer work. Run one expensive action at a time. Phase-A selector results and Phase-B/C assets are already archived; do not rerun them unless changing the protocol.

The CRN experiment below uses simulator-generated coalition utilities, not deterministic fixture functions. It compares fixed coalition differences under common random numbers and independent shocks. External statistics remain conditional on an audited per-user file.

In [23]:
RUN_ALL = False
RUN_CRN = True
RUN_EXTERNAL_STATS = False
RUN_SCALABILITY_REFERENCE = False

In [24]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd

CANDIDATES = [Path.cwd(), Path.cwd() / 'paper-ideas' / 'CURE-Rec' / 'code', *Path.cwd().parents]
ROOT = next(p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from cure_rec.config import load_settings
from cure_rec.pipeline import run_experiment
from cure_rec.revision_suite import paired_user_statistics

FULL_CONFIG = ROOT / 'configs' / 'curesim_full.yaml'
RESULTS = ROOT / "results" / "reviewer_phase_assets"
print('CURE-Rec root:', ROOT)

CURE-Rec root: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code


## Controls

Keep all flags false except the single action you are intentionally running.

In [25]:
RUN_CRN = True
RUN_EXTERNAL_STATS = False
RUN_SCALABILITY_REFERENCE = False
CRN_SEEDS = (300, 301, 302, 303, 304)
assert sum((RUN_CRN, RUN_EXTERNAL_STATS, RUN_SCALABILITY_REFERENCE)) <= 1

## Phase B/C — already archived

Inspect these files instead of rerunning the full game:

```text
results/reviewer_phase_assets/tables/phase_b_objective_constraint_ablation.csv
results/reviewer_phase_assets/tables/phase_c_sampled_shapley_fidelity.csv
results/reviewer_phase_assets/revision_manifest.json
```

In [26]:
for path in [RESULTS / 'tables' / 'phase_b_objective_constraint_ablation.csv', RESULTS / 'tables' / 'phase_c_sampled_shapley_fidelity.csv']:
    if path.exists():
        print('\n', path.name)
        display(pd.read_csv(path))
    else:
        print('Missing:', path)


 phase_b_objective_constraint_ablation.csv


,objective,constraint_mode,penalty,mask,interventions,feasible,runtime_seconds,cost,relevance_delta_lower,provider_disparity_upper,fatigue_upper
0,maximin,hard,0.00,1,repeat_cap,True,0.000630,0.05,-0.041478,0.244275,0.0
1,maximin,penalty,0.00,1,repeat_cap,True,0.000642,0.05,-0.041478,0.244275,0.0
2,maximin,penalty,0.25,1,repeat_cap,True,0.000623,0.05,-0.041478,0.244275,0.0
3,maximin,penalty,0.50,1,repeat_cap,True,0.000622,0.05,-0.041478,0.244275,0.0
4,maximin,penalty,1.00,1,repeat_cap,True,0.000626,0.05,-0.041478,0.244275,0.0
5,maximin,penalty,2.00,1,repeat_cap,True,0.000669,0.05,-0.041478,0.244275,0.0
6,maximin,penalty,5.00,1,repeat_cap,True,0.000642,0.05,-0.041478,0.244275,0.0
7,maximin,penalty,10.00,1,repeat_cap,True,0.000617,0.05,-0.041478,0.244275,0.0
8,mean,hard,0.00,1,repeat_cap,True,0.000866,0.05,-0.041478,0.244275,0.0
9,mean,penalty,0.00,1,repeat_cap,True,0.000879,0.05,-0.041478,0.244275,0.0



 phase_c_sampled_shapley_fidelity.csv


,budget,mae,max_error,sign_agreement,rank_correlation,runtime_seconds
0,32,0.000849,0.001590,1.0,1.0,0.000713
1,128,0.001550,0.003293,1.0,1.0,0.000657
2,512,0.000502,0.000822,1.0,1.0,0.002121
3,2048,0.000207,0.000377,1.0,1.0,0.007253


## Phase C — optional larger-player reference

The current exact implementation is six-player. Do not claim 8/10-player scalability until a distinct player library is implemented. This cell only records the required protocol and intentionally does not create duplicate players.

In [27]:
if RUN_SCALABILITY_REFERENCE:
    raise NotImplementedError('Add distinct operational players before running n=8 or n=10; duplicate players are prohibited.')
print('Scalability reference disabled unless a distinct 8/10-player library is available.')

Scalability reference disabled unless a distinct 8/10-player library is available.


## CRN paired coalition-difference study

Fixed pairs:

- empty coalition (0) versus `repeat_cap` (1)
- `repeat_cap` (1) versus `repeat_cap + tail_slot` (5)

For each seed, the exact simulator game is run with CRN enabled and disabled. The reported unit is the paired coalition difference, not the standard deviation of selected policies.

In [28]:
def game_pair_difference(settings, seed, mask_a, mask_b, common_random_numbers):
    cfg = settings.model_copy(deep=True)
    cfg.run.seed = int(seed)
    cfg.run.common_random_numbers = bool(common_random_numbers)
    cfg.run.name = f'crn-{common_random_numbers}-{seed}-{mask_a}-{mask_b}'
    cfg.run.output_root = ROOT / 'runs' / 'reviewer-crn'
    logger, game, _ = run_experiment(cfg)
    scenario_diffs = []
    for scenario in game.scenario_games.values():
        a = scenario.values[mask_a].utility
        b = scenario.values[mask_b].utility
        scenario_diffs.append(float(b - a))
    return float(np.mean(scenario_diffs)), logger.run_dir

def run_crn_study(seeds=CRN_SEEDS):
    settings = load_settings(FULL_CONFIG)
    rows = []
    for pair in ((0, 1), (1, 5)):
        for seed in seeds:
            crn_diff, crn_dir = game_pair_difference(settings, seed, pair[0], pair[1], True)
            iid_diff, iid_dir = game_pair_difference(settings, seed, pair[0], pair[1], False)
            rows.append({'seed': seed, 'mask_a': pair[0], 'mask_b': pair[1], 'crn_difference': crn_diff, 'independent_difference': iid_diff, 'crn_run': str(crn_dir), 'independent_run': str(iid_dir)})
    frame = pd.DataFrame(rows)
    frame['paired_difference'] = frame['crn_difference'] - frame['independent_difference']
    summary = frame.groupby(['mask_a','mask_b']).agg(crn_variance=('crn_difference','var'), independent_variance=('independent_difference','var'), crn_mean=('crn_difference','mean'), independent_mean=('independent_difference','mean'), n=('seed','count')).reset_index()
    summary['variance_ratio'] = summary['crn_variance'] / summary['independent_variance']
    out = ROOT / "results" / "reviewer_phase_assets" / "crn_simulator"
    out.mkdir(parents=True, exist_ok=True)
    frame.to_csv(out / 'crn_paired_differences.csv', index=False)
    summary.to_csv(out / 'crn_summary.csv', index=False)
    (out / 'crn_manifest.json').write_text(json.dumps({'seeds': list(seeds), 'pairs': [[0,1],[1,5]], 'claim_scope': 'CURE-Sim paired coalition differences', 'common_random_numbers_definition': 'same seed and shared shock stream within each exact game', 'independent_definition': 'coalition-specific shock offset in simulator'}, indent=2))
    return frame, summary

if RUN_CRN:
    crn_rows, crn_summary = run_crn_study()
    display(crn_summary)
else:
    print('Simulator CRN study disabled.')

2026-08-15 05:40:17,317 | INFO | run_started | {"config_hash": "bc47c0331c5b17b5", "run_id": "crn-True-300-0-1-20260815T044017Z-d5369e69"}
2026-08-15 05:40:17,317 | INFO | exact_game_started | {}
2026-08-15 05:40:17,319 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-15 05:43:04,402 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.1406521512316774, "scenario": "nominal", "shapley_efficiency_gap": 0.0}
2026-08-15 05:43:04,404 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-15 05:47:13,362 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.12824569278118791, "scenario": "fatigue_stress", "shapley_efficiency_gap": 0.0}
2026-08-15 05:47:13,364 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-15 05:50:57,995 | INFO | scenario_game_completed | {"grand_coalition_impr

,mask_a,mask_b,crn_variance,independent_variance,crn_mean,independent_mean,n,variance_ratio
0,0,1,1.791225e-06,1.791225e-06,0.303209,0.303209,5,1.0
1,1,5,1.236512e-07,1.236512e-07,-0.076364,-0.076364,5,1.0


## Phase D — paired external statistics

Only run this after producing an audited per-user table with columns `user_id`, `model`, `hit`, and `ndcg`. This is ranking uncertainty, not causal policy evidence.

In [31]:
if RUN_EXTERNAL_STATS:
    metrics_path = RESULTS / 'per_user_metrics.csv'
    if not metrics_path.exists():
        raise FileNotFoundError(f'Create an audited per-user file first: {metrics_path}')
    metrics = pd.read_csv(metrics_path)
    required = {'user_id', 'model', 'hit', 'ndcg'}
    missing = required - set(metrics.columns)
    if missing: raise ValueError(f'Missing required columns: {sorted(missing)}')
    ci, tests = paired_user_statistics(metrics)
    ci.to_csv(RESULTS / 'paired_bootstrap_ci.csv', index=False)
    tests.to_csv(RESULTS / 'paired_tests_holm.csv', index=False)
    display(ci); display(tests)
else:
    print('External paired statistics disabled.')

External paired statistics disabled.


## Phase E — release checklist

Before manuscript updates, verify:

- CRN pair table and manifest are archived;
- exact six-player results remain primary;
- sampled Shapley is labelled approximate;
- external results remain descriptive ranking evidence;
- all paths in manuscript are relative;
- `REPRODUCE.md` and checksums include new assets;
- manuscript contains no unresolved `Table ??`, `???`, or stale metric notation.

In [21]:
if RUN_ALL:
    manifest = RESULTS / 'remaining_actions_run_all_manifest.json'
    manifest.write_text(json.dumps({'run_all': True, 'phase_a': 'archived_not_rerun', 'phase_bc': 'archived_inspected', 'crn': 'executed', 'phase_d': 'executed' if RUN_EXTERNAL_STATS else 'skipped_missing_per_user_metrics', 'scalability': 'skipped_no_distinct_8_10_player_library', 'config_changed': False}, indent=2))
    print(manifest)


/Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code/results/reviewer_phase_assets/remaining_actions_run_all_manifest.json


In [22]:
RUN_ALL = False
RUN_CRN = True
RUN_EXTERNAL_STATS = False
RUN_SCALABILITY_REFERENCE = False